# 5-Node Graph Toy Experiment: Joint Node + Edge Diffusion

Extends `toy_experiment_graph.ipynb` from a pixel grid to small **irregular graphs**.
The key addition: the graph **topology (edges) is itself denoised jointly with node features**.

| Component | Grid GNN notebook | This notebook |
|---|---|---|
| Graph | 24×24 pixel grid (576 nodes, fixed edges) | 5-node cycle or line (variable edges) |
| Denoising target | Node features only | Node features **+** edges jointly |
| Boundary handling | Periodic `edge_index` (equivalent to circular padding) | N/A — small graphs have no boundary |
| Architecture | `GraphResNet` (fixed `edge_index`, scatter_add) | `GraphJointDenoiser` (dynamic adj, bmm) |

**What stays the same**: D3PM forward process, `cosine_alpha_bar` schedule, `x0`-parameterisation,
BCE loss, `q_posterior_probs` reverse sampling, local-majority violation metric.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
import matplotlib.pyplot as plt
import csv, os

In [ ]:
N_NODES         = 5
HIDDEN_DIM      = 64
T               = 500    # diffusion timesteps (matches existing notebooks)
EPOCHS          = 500
LR              = 2e-4   # matches existing notebooks
BATCH_SIZE      = 128
NSTEPS_VALID    = 400
NSTEPS_INVALID  = 3
NUM_LAYERS_LIST = [0, 1, 2, 3, 4]
DEVICE          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LOG_FILE        = 'results_log_graph5.csv'
OUT_DIR         = 'intermediate_graph5_results'
os.makedirs(OUT_DIR, exist_ok=True)
print('Device:', DEVICE)

## D3PM forward process
Identical to `toy_experiment_graph.ipynb` — `cosine_alpha_bar` with offset `s=0.008`,
and the exact D3PM posterior for reverse sampling.

In [ ]:
def cosine_alpha_bar(t, s=0.008):
    """Cumulative alpha_bar: 1 at t=0, ~0 at t=1. Matches existing notebooks."""
    f_t = torch.cos((t + s) / (1.0 + s) * (torch.pi / 2.0)) ** 2
    f_0 = torch.cos((torch.zeros_like(t) + s) / (1.0 + s) * (torch.pi / 2.0)) ** 2
    return (f_t / f_0.clamp(min=1e-8)).clamp(0.0, 1.0)


def q_sample(x0_float, t_float):
    """
    Forward process: corrupt x0 -> x_t.
    x0_float: float binary tensor of any shape (B, ...)
    t_float:  (B,) timestep in (0, 1]
    """
    ab = cosine_alpha_bar(t_float).to(x0_float.device)
    for _ in range(x0_float.dim() - 1):
        ab = ab.unsqueeze(-1)
    ab = ab.expand_as(x0_float)
    keep  = torch.bernoulli(ab).bool()
    noise = torch.randint(0, 2, x0_float.shape, device=x0_float.device).float()
    return torch.where(keep, x0_float, noise)


def q_posterior_probs(x_t, x0, t_float, t_prev_float):
    """
    D3PM posterior: P(x_{t-1}=1 | x_t, x0) for binary uniform noise.
    Works for any tensor shape — used for both nodes and edges.
    Matches the formula in toy_experiment_graph.ipynb.
    """
    ab_t    = cosine_alpha_bar(t_float).to(x_t.device)
    ab_prev = cosine_alpha_bar(t_prev_float).to(x_t.device)
    for _ in range(x_t.dim() - 1):
        ab_t    = ab_t.unsqueeze(-1)
        ab_prev = ab_prev.unsqueeze(-1)

    alpha_t = (ab_t / ab_prev.clamp(min=1e-8)).clamp(0.0, 1.0)
    beta_t  = 1.0 - alpha_t
    xt = x_t.float(); x0 = x0.float()

    qt_c0  = (1.0 - beta_t) * (1.0 - xt) + (beta_t / 2) * xt
    qt_c1  = (beta_t / 2)   * (1.0 - xt) + (1.0 - beta_t) * xt
    qb_c0  = (1.0 + ab_prev) / 2.0 * (1.0 - x0) + (1.0 - ab_prev) / 2.0 * x0
    qb_c1  = (1.0 - ab_prev) / 2.0 * (1.0 - x0) + (1.0 + ab_prev) / 2.0 * x0

    theta0 = qt_c0 * qb_c0
    theta1 = qt_c1 * qb_c1
    return theta1 / (theta0 + theta1 + 1e-8)

## Training data
- **Cycle**: 0–1–2–3–4–0, all nodes white (1). Each node degree = 2.
- **Line**: 0–1–2–3–4, all nodes black (0). Endpoint degree = 1, middle degree = 2.

The local consistency rule (same as grid): a node is consistent if its feature agrees
with the majority of its neighbours' features.

In [ ]:
def make_cycle():
    adj = torch.zeros(N_NODES, N_NODES)
    for i in range(N_NODES):
        j = (i + 1) % N_NODES
        adj[i, j] = adj[j, i] = 1.0
    return torch.ones(N_NODES), adj   # all white


def make_line():
    adj = torch.zeros(N_NODES, N_NODES)
    for i in range(N_NODES - 1):
        adj[i, i+1] = adj[i+1, i] = 1.0
    return torch.zeros(N_NODES), adj  # all black


def sample_batch(batch_size, device):
    nfs, adjs = [], []
    for _ in range(batch_size):
        nf, adj = make_cycle() if torch.rand(1) > 0.5 else make_line()
        nfs.append(nf); adjs.append(adj)
    return torch.stack(nfs).to(device), torch.stack(adjs).to(device)


def symmetrise_adj(adj):
    """Enforce symmetry and zero diagonal."""
    adj = ((adj + adj.transpose(-1, -2)) / 2 >= 0.5).float()
    adj.diagonal(dim1=-1, dim2=-2).fill_(0)
    return adj


# Visualise training data
def draw_graph(node_x, adj_x, ax, title='', violations=None):
    angles = [2 * np.pi * i / N_NODES - np.pi / 2 for i in range(N_NODES)]
    pos    = [(np.cos(a), np.sin(a)) for a in angles]
    for i in range(N_NODES):
        for j in range(i + 1, N_NODES):
            val = adj_x[i, j].item() if hasattr(adj_x[i, j], 'item') else float(adj_x[i, j])
            if val > 0.5:
                ax.plot([pos[i][0], pos[j][0]], [pos[i][1], pos[j][1]],
                        color='#444', lw=2, zorder=1)
    for i in range(N_NODES):
        feat  = node_x[i].item() if hasattr(node_x[i], 'item') else float(node_x[i])
        color = 'white' if feat > 0.5 else 'black'
        viol  = violations is not None and violations[i]
        circle = plt.Circle(pos[i], 0.18, color=color,
                             ec='red' if viol else '#555',
                             lw=2.5 if viol else 1.5, zorder=2)
        ax.add_patch(circle)
    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(title, fontsize=9)


nf_c, adj_c = make_cycle()
nf_l, adj_l = make_line()
fig, axes = plt.subplots(1, 2, figsize=(5, 3))
draw_graph(nf_c, adj_c, axes[0], 'Cycle (all white)')
draw_graph(nf_l, adj_l, axes[1], 'Line (all black)')
plt.suptitle('Training data', fontsize=11)
plt.tight_layout()
plt.savefig('toy_graph5_training_data.png', dpi=120, bbox_inches='tight')
plt.show()

## Architecture

`GraphJointDenoiser` follows the same design as `GraphResNet` in the grid GNN notebook:
- `TimestepEmbedding` with projection (identical class)
- `num_layers` message-passing layers — here using `bmm` on a **dynamic noisy adjacency**
  rather than `scatter_add` on a fixed `edge_index`
- One extra timestep embedding after all layers (like `add_one=True` in `MinimalResNet`)
- **New**: an edge decoder that predicts edge logits from pairs of node representations

Input normalisation: `{0,1} → {-1, +1}` (identical to existing notebooks).

In [ ]:
class TimestepEmbedding(nn.Module):
    """Sinusoidal embedding + projection. Identical to toy_experiment_graph.ipynb."""
    def __init__(self, emb_dim):
        super().__init__()
        self.emb_dim = emb_dim
        self.proj    = nn.Sequential(nn.Linear(emb_dim, emb_dim), nn.ReLU())

    def forward(self, t):
        d     = self.emb_dim // 2
        freqs = 10000 ** (torch.arange(d, device=t.device).float() / max(d - 1, 1))
        args  = t[:, None] / freqs[None, :]
        emb   = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
        return self.proj(emb)


class DynamicMessagePassingLayer(nn.Module):
    """
    Like MessagePassingLayer in the grid notebook but uses a dynamic
    (noisy, changing) adjacency matrix via bmm instead of a fixed edge_index.
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.linear = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, h, adj_x, t_emb):
        """h: (B,N,D)  adj_x: (B,N,N)  t_emb: (B,D)"""
        B, N, D = h.shape
        deg  = adj_x.sum(-1, keepdim=True).clamp(min=1)
        agg  = torch.bmm(adj_x, h) / deg          # mean-neighbour repr
        t_b  = t_emb.unsqueeze(1).expand(B, N, D)
        delta = F.relu(self.linear(h + agg + t_b))
        return h + delta


class GraphJointDenoiser(nn.Module):
    """
    Jointly denoises node features and edges.
    Mirrors GraphResNet but also outputs edge logits.
    Has num_layers+1 TimestepEmbeddings (like add_one=True in MinimalResNet).
    """
    def __init__(self, n_nodes=N_NODES, hidden_dim=HIDDEN_DIM, num_layers=1):
        super().__init__()
        self.num_layers = num_layers
        self.up     = nn.Linear(1, hidden_dim)
        # num_layers+1 embeddings: one per MP layer + one final (like add_one)
        self.t_embs = nn.ModuleList(
            [TimestepEmbedding(hidden_dim) for _ in range(num_layers + 1)])
        self.layers = nn.ModuleList(
            [DynamicMessagePassingLayer(hidden_dim) for _ in range(num_layers)])
        self.node_out = nn.Linear(hidden_dim, 1)
        self.edge_out = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, t, node_x_norm, adj_x):
        """
        t:           (B,)      timestep in [0,1]
        node_x_norm: (B, N, 1) in {-1, +1}  (normalised from {0,1})
        adj_x:       (B, N, N) noisy binary adjacency
        Returns: node_logits (B,N,1), edge_logits (B,N,N,1)
        """
        B, N, _ = node_x_norm.shape
        h = self.up(node_x_norm)   # (B, N, D)

        for i, layer in enumerate(self.layers):
            t_emb = self.t_embs[i](t)
            h = layer(h, adj_x, t_emb)

        # Final timestep embedding added before output (add_one analogue)
        h = h + self.t_embs[-1](t).unsqueeze(1).expand(B, N, -1)

        node_logits = self.node_out(h)                     # (B, N, 1)

        h_i = h.unsqueeze(2).expand(-1, -1, N, -1)        # (B, N, N, D)
        h_j = h.unsqueeze(1).expand(-1, N, -1, -1)        # (B, N, N, D)
        edge_logits = self.edge_out(torch.cat([h_i, h_j], dim=-1))  # (B,N,N,1)
        edge_logits = (edge_logits + edge_logits.transpose(1, 2)) / 2  # symmetrize

        return node_logits, edge_logits

## Training
Same structure as the grid GNN notebook:
1. Sample random `t`, corrupt both nodes and edges with `q_sample`
2. Normalise noisy inputs to `{-1, +1}`
3. Minimise BCE loss between predicted and true `x0` for both nodes and edges

In [ ]:
def train_model(model, epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE,
                nsteps=T, device=DEVICE):
    model     = model.to(device)
    opt       = Adam(model.parameters(), lr=lr)
    triu      = torch.triu(torch.ones(N_NODES, N_NODES, device=device), diagonal=1).bool()
    final_loss = 0.0

    for epoch in range(epochs):
        model.train()
        node_x0, adj_x0 = sample_batch(batch_size, device)  # float {0,1}

        t_int   = torch.randint(1, nsteps + 1, (batch_size,), device=device)
        t_float = t_int.float() / nsteps

        # Corrupt nodes and edges independently
        node_xt = q_sample(node_x0, t_float)                  # (B, N)
        adj_xt  = symmetrise_adj(q_sample(adj_x0, t_float))   # (B, N, N)

        # Normalise to {-1, +1} — matches existing notebooks
        node_xt_norm = (node_xt * 2.0 - 1.0).unsqueeze(-1)   # (B, N, 1)

        node_logits, edge_logits = model(t_float, node_xt_norm, adj_xt)

        node_loss = F.binary_cross_entropy_with_logits(
            node_logits.squeeze(-1), node_x0
        )
        # Upper-triangle only to avoid double-counting symmetric edges
        edge_loss = F.binary_cross_entropy_with_logits(
            edge_logits.squeeze(-1)[:, triu], adj_x0[:, triu]
        )
        loss = node_loss + edge_loss
        opt.zero_grad(); loss.backward(); opt.step()
        final_loss = loss.item()

        if (epoch + 1) % 100 == 0:
            print(f'  Epoch {epoch+1:>4}/{epochs}  loss={final_loss:.4f}')

    model.eval()
    return model, final_loss

## Sampling
Uses the exact D3PM posterior `q_posterior_probs` — identical to the grid GNN notebook.
Applied separately to nodes and edges; edges are symmetrised after each step.

In [ ]:
@torch.no_grad()
def sample_graphs(model, batch_size=4, nsteps=NSTEPS_VALID, device=DEVICE):
    model.eval().to(device)

    node_x = torch.randint(0, 2, (batch_size, N_NODES), device=device).float()
    adj_x  = symmetrise_adj(
        torch.randint(0, 2, (batch_size, N_NODES, N_NODES), device=device).float()
    )

    for i in range(nsteps, 0, -1):
        t_float = torch.full((batch_size,), i / nsteps, device=device)
        t_prev  = torch.full((batch_size,), (i - 1) / nsteps, device=device)

        node_x_norm = (node_x * 2.0 - 1.0).unsqueeze(-1)  # {-1, +1}
        node_logits, edge_logits = model(t_float, node_x_norm, adj_x)

        p_node_1 = torch.sigmoid(node_logits.squeeze(-1))  # (B, N)
        p_edge_1 = torch.sigmoid(edge_logits.squeeze(-1))  # (B, N, N)

        if i == 1:
            node_x = (p_node_1 > 0.5).float()
            adj_x  = symmetrise_adj((p_edge_1 > 0.5).float())
        else:
            # D3PM posterior for nodes
            p_prev_node = (
                (1 - p_node_1) * q_posterior_probs(node_x, torch.zeros_like(node_x), t_float, t_prev)
              + p_node_1       * q_posterior_probs(node_x, torch.ones_like(node_x),  t_float, t_prev)
            )
            node_x = torch.bernoulli(p_prev_node.clamp(0, 1))

            # D3PM posterior for edges
            p_prev_edge = (
                (1 - p_edge_1) * q_posterior_probs(adj_x, torch.zeros_like(adj_x), t_float, t_prev)
              + p_edge_1       * q_posterior_probs(adj_x, torch.ones_like(adj_x),  t_float, t_prev)
            )
            adj_x = symmetrise_adj(torch.bernoulli(p_prev_edge.clamp(0, 1)))

    return node_x.cpu(), adj_x.cpu()

In [ ]:
@torch.no_grad()
def sample_graph_with_snapshots(model, nsteps=NSTEPS_VALID, n_snapshots=8, device=DEVICE):
    """Run a single graph denoising and capture snapshots at evenly-spaced steps."""
    model.eval().to(device)

    node_x = torch.randint(0, 2, (1, N_NODES), device=device).float()
    adj_x  = symmetrise_adj(
        torch.randint(0, 2, (1, N_NODES, N_NODES), device=device).float()
    )

    if nsteps <= 10:
        capture = set(range(nsteps, -1, -1))
    else:
        step_size = max(1, nsteps // (n_snapshots - 1))
        capture   = set(range(nsteps, 0, -step_size)) | {1}

    snapshots = [(nsteps, node_x[0].cpu().clone(), adj_x[0].cpu().clone())]

    for i in range(nsteps, 0, -1):
        t_float = torch.full((1,), i / nsteps, device=device)
        t_prev  = torch.full((1,), (i - 1) / nsteps, device=device)

        node_x_norm              = (node_x * 2.0 - 1.0).unsqueeze(-1)
        node_logits, edge_logits = model(t_float, node_x_norm, adj_x)

        p_node_1 = torch.sigmoid(node_logits.squeeze(-1))
        p_edge_1 = torch.sigmoid(edge_logits.squeeze(-1))

        if i == 1:
            node_x = (p_node_1 > 0.5).float()
            adj_x  = symmetrise_adj((p_edge_1 > 0.5).float())
        else:
            p_prev_node = (
                (1 - p_node_1) * q_posterior_probs(node_x, torch.zeros_like(node_x), t_float, t_prev)
              + p_node_1       * q_posterior_probs(node_x, torch.ones_like(node_x),  t_float, t_prev)
            )
            node_x = torch.bernoulli(p_prev_node.clamp(0, 1))

            p_prev_edge = (
                (1 - p_edge_1) * q_posterior_probs(adj_x, torch.zeros_like(adj_x), t_float, t_prev)
              + p_edge_1       * q_posterior_probs(adj_x, torch.ones_like(adj_x),  t_float, t_prev)
            )
            adj_x = symmetrise_adj(torch.bernoulli(p_prev_edge.clamp(0, 1)))

        if i in capture:
            snapshots.append((i - 1, node_x[0].cpu().clone(), adj_x[0].cpu().clone()))

    if len(snapshots) > n_snapshots:
        idx       = np.linspace(0, len(snapshots) - 1, n_snapshots, dtype=int)
        snapshots = [snapshots[j] for j in idx]

    return snapshots


def plot_graph_denoising(model, nsteps, title='', n_snapshots=8):
    """Run denoising and plot snapshots as a row of graph visualizations."""
    snapshots = sample_graph_with_snapshots(model, nsteps=nsteps, n_snapshots=n_snapshots)
    n         = len(snapshots)
    fig, axes = plt.subplots(1, n, figsize=(2.5 * n, 3.0))
    if n == 1:
        axes = [axes]

    for ax, (step, node_x, adj_x) in zip(axes, snapshots):
        is_final = (step == 0)
        viols    = None
        if is_final:
            _, _, masks = node_violations(node_x.unsqueeze(0), adj_x.unsqueeze(0))
            viols = masks[0]
        label = f't={step}' + (' (final)' if is_final else '')
        draw_graph(node_x, adj_x, ax, title=label, violations=viols)

    fig.suptitle(title, fontsize=10, y=1.02)
    plt.tight_layout()
    return fig


print('Denoising helpers defined.')

In [ ]:
def node_violations(node_x, adj_x):
    """
    Same local-majority rule as the grid experiments:
    a node violates if its feature disagrees with the majority of its neighbours.
    Isolated nodes (degree 0) always violate.
    Returns (total, per_sample_list, per_sample_masks).
    """
    per_sample, masks = [], []
    for b in range(node_x.shape[0]):
        mask, count = [], 0
        for i in range(N_NODES):
            nbrs = adj_x[b, i].nonzero(as_tuple=True)[0]
            if len(nbrs) == 0:
                mask.append(True); count += 1
            else:
                majority = (node_x[b, nbrs].mean() > 0.5).float()
                viol = (node_x[b, i] != majority).item()
                mask.append(bool(viol))
                if viol: count += 1
        per_sample.append(count); masks.append(mask)
    return sum(per_sample), per_sample, masks

## Experiment sweep
- `L=0`: no message passing — model cannot see neighbours or edges, cannot determine topology
- `L=1`: 1-hop — model sees immediate neighbours, can compute degree
- `L=2`: 2-hop — on a 5-node graph this covers the entire graph from any node

Expected: L=0 should fail (many violations even with valid sampling);
L=1 should succeed; L=2 may collapse (reach 0 invalid violations).

In [ ]:
if os.path.exists(LOG_FILE):
    with open(LOG_FILE) as f:
        done = {int(row['num_layers']) for row in csv.DictReader(f)}
    print(f'Already logged: L={sorted(done)}')
else:
    done = set()
    with open(LOG_FILE, 'w', newline='') as f:
        csv.writer(f).writerow(['num_layers', 'n_params', 'final_loss',
                                 'valid_violations_mean', 'invalid_violations_mean',
                                 'valid_violations_all',  'invalid_violations_all'])

for NUM_LAYERS in NUM_LAYERS_LIST:
    img_files = [
        os.path.join(OUT_DIR, f'toy_graph5_L{NUM_LAYERS}_valid{NSTEPS_VALID}_invalid{NSTEPS_INVALID}.png'),
        os.path.join(OUT_DIR, f'toy_graph5_L{NUM_LAYERS}_denoise_valid.png'),
        os.path.join(OUT_DIR, f'toy_graph5_L{NUM_LAYERS}_denoise_invalid.png'),
    ]
    images_exist   = all(os.path.exists(f) for f in img_files)
    already_logged = NUM_LAYERS in done

    if already_logged and images_exist:
        print(f'Skipping L={NUM_LAYERS} (logged + images present)'); continue

    print(f'\n{"="*50}')
    print(f'L={NUM_LAYERS}' + (' (retrain for images only)' if already_logged else ''))
    print(f'{"="*50}')

    model    = GraphJointDenoiser(num_layers=NUM_LAYERS)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'Parameters: {n_params:,}')

    model, final_loss = train_model(model)
    print(f'Final loss: {final_loss:.4f}')

    nv, av = sample_graphs(model, batch_size=4, nsteps=NSTEPS_VALID)
    ni, ai = sample_graphs(model, batch_size=4, nsteps=NSTEPS_INVALID)

    tot_v, list_v, masks_v = node_violations(nv, av)
    tot_i, list_i, masks_i = node_violations(ni, ai)
    mean_v = tot_v / 4;  mean_i = tot_i / 4
    print(f'Valid   violations ({NSTEPS_VALID} steps): {mean_v:.2f}  {list_v}')
    print(f'Invalid violations ({NSTEPS_INVALID} steps): {mean_i:.2f}  {list_i}')

    # ── Overview plot (training data + valid + invalid samples) ───────────────
    fig, axes = plt.subplots(3, 4, figsize=(10, 8))
    draw_graph(nf_c, adj_c, axes[0][0], 'Cycle (train)')
    draw_graph(nf_l, adj_l, axes[0][1], 'Line (train)')
    axes[0][2].axis('off'); axes[0][3].axis('off')
    for col in range(4):
        draw_graph(nv[col], av[col], axes[1][col],
                   f'Valid #{col+1} ({list_v[col]})', violations=masks_v[col])
        draw_graph(ni[col], ai[col], axes[2][col],
                   f'Invalid #{col+1} ({list_i[col]})', violations=masks_i[col])
    axes[0][0].set_ylabel('Training data', fontsize=9)
    axes[1][0].set_ylabel(f'Valid ({NSTEPS_VALID} steps)', fontsize=9)
    axes[2][0].set_ylabel(f'Invalid ({NSTEPS_INVALID} steps)', fontsize=9)
    plt.suptitle(f'5-node joint diffusion  |  L={NUM_LAYERS}  |  '
                 f'valid={mean_v:.2f}  invalid={mean_i:.2f}', fontsize=11)
    plt.tight_layout()
    fname_overview = img_files[0]
    plt.savefig(fname_overview, dpi=150, bbox_inches='tight')
    print(f'Saved {fname_overview}')
    plt.show()

    # ── Denoising trajectory: valid (400 steps, 8 snapshots) ──────────────────
    fig_v = plot_graph_denoising(
        model, nsteps=NSTEPS_VALID, n_snapshots=8,
        title=f'5-node joint diffusion  L={NUM_LAYERS}  —  denoising ({NSTEPS_VALID} steps)')
    fig_v.savefig(img_files[1], dpi=150, bbox_inches='tight')
    print(f'Saved {img_files[1]}')
    plt.show()

    # ── Denoising trajectory: invalid (3 steps, all steps shown) ──────────────
    fig_i = plot_graph_denoising(
        model, nsteps=NSTEPS_INVALID, n_snapshots=NSTEPS_INVALID + 1,
        title=f'5-node joint diffusion  L={NUM_LAYERS}  —  denoising ({NSTEPS_INVALID} steps)')
    fig_i.savefig(img_files[2], dpi=150, bbox_inches='tight')
    print(f'Saved {img_files[2]}')
    plt.show()

    if not already_logged:
        with open(LOG_FILE, 'a', newline='') as f:
            csv.writer(f).writerow([NUM_LAYERS, n_params, f'{final_loss:.4f}',
                                      f'{mean_v:.2f}', f'{mean_i:.2f}',
                                      str(list_v), str(list_i)])
        done.add(NUM_LAYERS)